In [1]:
from IPython.display import clear_output
from Fetch_Live_Data import *
from Trade_Selection_All import *
from Chandelier_ZLSMA import *
from Trade_Execution import *

In [2]:
def pick_best_coin():
    scalping_filter = BinanceAllUSDTScalpingFilter(
        max_workers=8,
        delay_between_requests=0.1,
        weight_profile='volatile'  # Options: 'balanced', 'volatile', 'trending'
    )

    print("Scanning ALL Binance USDT pairs for scalping opportunities...")

    # get a list of dicts (or records)
    best_coins = scalping_filter.filter_all_usdt_pairs(
        min_volume=50_000,
        top_n=30,
        use_parallel=True,
        volume_filter_first=False,
        use_percentile_volume=True
    )

    # turn into a DataFrame
    df = pd.DataFrame(best_coins)

    # base filters: uptrend + cheap coins
    base_mask = (df['trend_direction'] == 1) & (df['current_price'] <= 50)

    # try successive RSI thresholds
    for rsi_limit in (40, 50, 60, 65):
        candidates = df[base_mask & (df['rsi'] <= rsi_limit)]
        if not candidates.empty:
            # return the single symbol with highest scalping_score
            return candidates.nlargest(1, 'buy_pressure')['symbol'].iloc[0]

    # nothing matched
    return None

In [ ]:
# Define your function to fetch, calculate, and merge the data
def fetch_and_process_data(symbol):
    df = get_single_fetch(symbol, 500)  # Fetch data
    df_zlsma = calculate_zlsma(df, close_column='close', length=200)  # Calculate ZLSMA
    df_chandelier = calculate_chandelier(df, atr_period=1, atr_multiplier=2.0)  # Calculate Chandelier
    merged_chandelier_zlsma = merge_zlsma_chandelier(df_zlsma, df_chandelier)  # Merge ZLSMA and Chandelier
    merged_chandelier_zlsma = merged_chandelier_zlsma[["timestamp", "close", "zlsma_200", "buy_signal", "sell_signal"]]  # Filter relevant columns
    return merged_chandelier_zlsma[480:]

def run_task():
    # Get the latest symbols every 2 hours
    symbols = pick_best_coin()
    return symbols

def fetch_and_display_data_(symbols):
    # Run the fetch_and_process_data every 10 seconds to display results
    result = fetch_and_process_data(symbols)
    clear_output(wait=True)  # Clear previous output in Jupyter Notebook
    print("Monitoring on: ", symbols)
    print(result)  # Display the new result
    print("\n")
    
    return result

# Main loop that runs every 2 hours
while True:
    symbols = run_task()  # Get the symbols every 2 hours
    print("New symbols received. Monitoring begins...\n")
    
    # Continuous 10-second updates with fetched data
    while True:
        out = fetch_and_display_data_(symbols)
        
        # Check if the DataFrame is not empty and get the last row
        if not out.empty:
            # Access the last row using iloc[-1]
            if out['close'].iloc[-1] > out['zlsma_200'].iloc[-1] and out['buy_signal'].iloc[-1] == True:
                print("Buy signal detected for symbols:", symbols)
                
                if out['buy_signal'].sum() <= 5:
                    #bot = SimpleATRTradingBot()
                    #result = bot.buy_signal(symbols, 10)
                    #status = bot.get_position_status()
                    print("Trade Taken:", symbols)
                else:
                    break
                    print("Refreshing for new symbols...")
            else:
                print("No buy signal detected for symbols:", symbols)
        else:
            print("The DataFrame is empty. No data available.")
        
        time.sleep(10)  # Wait for 10 seconds before running the next iteration
    
    # Sleep for 2 hours before getting new symbols (default behavior)
    time.sleep(2 * 60 * 60)  # Sleep for 2 hours (in seconds)

Monitoring on:  JSTUSDT
              timestamp    close  zlsma_200  buy_signal  sell_signal
480 2025-06-13 16:00:00  0.03271   0.031824           1            0
481 2025-06-13 16:15:00  0.03272   0.031835           1            0
482 2025-06-13 16:30:00  0.03271   0.031846           1            0
483 2025-06-13 16:45:00  0.03271   0.031857           1            0
484 2025-06-13 17:00:00  0.03278   0.031872           1            0
485 2025-06-13 17:15:00  0.03275   0.031886           1            0
486 2025-06-13 17:30:00  0.03271   0.031898           1            0
487 2025-06-13 17:45:00  0.03269   0.031910           1            0
488 2025-06-13 18:00:00  0.03267   0.031922           1            0
489 2025-06-13 18:15:00  0.03270   0.031936           1            0
490 2025-06-13 18:30:00  0.03259   0.031946           0            1
491 2025-06-13 18:45:00  0.03259   0.031957           0            1
492 2025-06-13 19:00:00  0.03257   0.031968           0            1
493 2025-0

KeyboardInterrupt: 

In [ ]:
bot = SimpleATRTradingBot()
result = bot.buy_signal('BTC', 1000)
status = bot.get_position_status()